# Steering Calibration Notebook

This notebook calibrates the JetBot's steering by:
1. Reading the IMU/gyroscope angle after a move
2. Applying corrective motor commands to straighten the robot
3. Repeating random turns + correction exercises many times
4. Saving a calibration map (`steering_calibration.json`) for later use

> **Place the robot on the track before running.** The robot will move during calibration.

## 1. Imports & Setup

In [ ]:
import time
import json
import random
import numpy as np
import ipywidgets as widgets
from IPython.display import display

from jetbot import Robot

robot = Robot()
print('Robot initialised.')

## 2. IMU Helper – Read Current Heading

JetBot carries an **MPU-6050** IMU on I2C bus 1.  
We integrate the gyroscope Z-axis to estimate yaw (heading drift).  
Replace `read_gyro_z()` if your board uses a different sensor.

In [ ]:
import smbus2

MPU_ADDR    = 0x68
PWR_MGMT_1  = 0x6B
GYRO_ZOUT_H = 0x47
GYRO_SCALE  = 131.0  # LSB/(deg/s) – default +/-250 deg/s range

bus = smbus2.SMBus(1)
bus.write_byte_data(MPU_ADDR, PWR_MGMT_1, 0)  # wake up the IMU

def read_gyro_z():
    """Return gyro Z rate in degrees/second."""
    high = bus.read_byte_data(MPU_ADDR, GYRO_ZOUT_H)
    low  = bus.read_byte_data(MPU_ADDR, GYRO_ZOUT_H + 1)
    raw  = (high << 8) | low
    if raw > 32767:
        raw -= 65536
    return raw / GYRO_SCALE

def integrate_yaw(duration_s, dt=0.01):
    """Integrate gyro Z over `duration_s` seconds and return total yaw in degrees."""
    yaw = 0.0
    for _ in range(int(duration_s / dt)):
        yaw += read_gyro_z() * dt
        time.sleep(dt)
    return yaw

print(f'Gyro Z at rest: {read_gyro_z():.2f} deg/s  (should be close to 0)')

## 3. Calibration Parameters

In [ ]:
base_speed_slider     = widgets.FloatSlider(value=0.25, min=0.05, max=0.6,   step=0.01,  description='Base speed')
move_duration_slider  = widgets.FloatSlider(value=0.4,  min=0.1,  max=2.0,   step=0.05,  description='Move time (s)')
correction_gain_slider= widgets.FloatSlider(value=0.015,min=0.001,max=0.1,   step=0.001, description='Corr. gain', readout_format='.3f')
num_trials_slider     = widgets.IntSlider(  value=20,   min=5,    max=60,    step=1,     description='# Trials')

display(base_speed_slider, move_duration_slider, correction_gain_slider, num_trials_slider)

STRAIGHT_TOL_DEG    = 2.0   # degrees – acceptable residual yaw
MAX_CORRECTION_ITERS = 8    # max correction pulses per trial

## 4. Movement Primitives

In [ ]:
def random_turn(base_speed, duration):
    """Drive with a random left/right bias, return (left, right) values used."""
    bias  = random.uniform(-0.25, 0.25)
    left  = float(np.clip(base_speed + bias, 0.05, 1.0))
    right = float(np.clip(base_speed - bias, 0.05, 1.0))
    robot.left_motor.value  = left
    robot.right_motor.value = right
    time.sleep(duration)
    robot.stop()
    return left, right


def correction_pulse(yaw_error_deg, base_speed, gain):
    """
    Counter-steer to reduce yaw error:
      +yaw  = turned right  -> steer left  (reduce left, boost right)
      -yaw  = turned left   -> steer right
    Returns (left, right) values applied.
    """
    steering = gain * yaw_error_deg
    left  = float(np.clip(base_speed - steering, 0.0, 1.0))
    right = float(np.clip(base_speed + steering, 0.0, 1.0))
    robot.left_motor.value  = left
    robot.right_motor.value = right
    time.sleep(0.15)
    robot.stop()
    time.sleep(0.05)
    return left, right


print('Movement helpers ready.')

## 5. Run Calibration Loop

Each trial:
1. **Random turn** – robot veers off straight
2. **Measure yaw** – gyro integration for 0.3 s
3. **Correction loop** – repeated short pulses until |yaw| < tolerance
4. **Record** everything for analysis

In [ ]:
calibration_records = []

N        = num_trials_slider.value
progress = widgets.IntProgress(value=0, min=0, max=N, description='Progress:', bar_style='info')
status   = widgets.Label(value='Starting…')
display(widgets.HBox([progress, status]))

for trial in range(N):
    status.value = f'Trial {trial+1}/{N} – random turn…'

    # Phase 1 ── random turn
    l_turn, r_turn = random_turn(base_speed_slider.value, move_duration_slider.value)
    time.sleep(0.1)

    # Measure resulting yaw
    yaw         = integrate_yaw(0.3)
    initial_yaw = yaw
    status.value = f'Trial {trial+1}/{N} – yaw={yaw:.1f} deg, correcting…'

    # Phase 2 ── correction loop
    iters        = 0
    corr_history = []

    while abs(yaw) > STRAIGHT_TOL_DEG and iters < MAX_CORRECTION_ITERS:
        l_c, r_c = correction_pulse(yaw, base_speed_slider.value, correction_gain_slider.value)
        time.sleep(0.1)
        yaw = integrate_yaw(0.3)
        corr_history.append({
            'yaw_before': round(yaw, 2),
            'left':  round(l_c, 4),
            'right': round(r_c, 4),
        })
        iters += 1

    calibration_records.append({
        'trial':           trial + 1,
        'left_turn':       round(l_turn, 4),
        'right_turn':      round(r_turn, 4),
        'initial_yaw_deg': round(initial_yaw, 2),
        'final_yaw_deg':   round(yaw, 2),
        'iters':           iters,
        'converged':       abs(yaw) <= STRAIGHT_TOL_DEG,
        'correction_gain': correction_gain_slider.value,
        'corrections':     corr_history,
    })

    status.value  = f'Trial {trial+1}/{N} – done  (final yaw={yaw:.1f} deg)'
    progress.value = trial + 1
    time.sleep(0.3)

robot.stop()
status.value = 'Calibration complete!'
print(f'\nCompleted {N} trials.')

## 6. Analyse & Fit a Steering Gain

In [ ]:
converged = [r for r in calibration_records if r['converged']]
avg_iters = np.mean([r['iters'] for r in calibration_records])
avg_final = np.mean([abs(r['final_yaw_deg']) for r in calibration_records])

print(f'Trials run        : {len(calibration_records)}')
print(f'Converged         : {len(converged)}  ({100*len(converged)/len(calibration_records):.0f}%)')
print(f'Avg corrections   : {avg_iters:.1f}')
print(f'Avg final |yaw|   : {avg_final:.2f} deg')

# Fit: motor differential vs yaw error to derive a data-driven steering gain
yaw_in, diff_out = [], []
for r in calibration_records:
    for c in r['corrections']:
        yaw_in.append(c['yaw_before'])
        diff_out.append(c['left'] - c['right'])

if len(yaw_in) > 3:
    coeffs       = np.polyfit(yaw_in, diff_out, 1)
    learned_gain = float(coeffs[0])
    print(f'\nFitted steering_gain = {learned_gain:.6f}  (motor diff per degree of yaw)')
else:
    learned_gain = correction_gain_slider.value
    print('Too few data points for fitting – using manual correction_gain.')

## 7. Save Calibration to File

In [ ]:
CALIB_PATH = 'steering_calibration.json'

output = {
    'version':           1,
    'description':       'JetBot steering calibration – gyro yaw correction',
    # ── Parameters ready to drop into a PD controller ──────────────────────
    'base_speed':        base_speed_slider.value,
    'steering_gain':     round(learned_gain, 6),
    'correction_gain':   correction_gain_slider.value,
    'straight_tol_deg':  STRAIGHT_TOL_DEG,
    # ── Summary ────────────────────────────────────────────────────────────
    'trials':            len(calibration_records),
    'converge_rate':     round(len(converged) / len(calibration_records), 3),
    'avg_corrections':   round(float(avg_iters), 2),
    'avg_final_yaw_deg': round(float(avg_final), 3),
    # ── Full log ───────────────────────────────────────────────────────────
    'records':           calibration_records,
}

with open(CALIB_PATH, 'w') as f:
    json.dump(output, f, indent=2)

print(f'Saved -> {CALIB_PATH}')
print(f'  steering_gain = {output["steering_gain"]}')
print(f'  base_speed    = {output["base_speed"]}')

## 8. Loading the Calibration in Another Notebook

Copy the cell below into `live_demo_executelock.ipynb` (or any road-following notebook)  
and use `STEERING_GAIN` in place of a hard-coded slider value.

In [ ]:
import json

with open('steering_calibration.json') as f:
    calib = json.load(f)

STEERING_GAIN = calib['steering_gain']
BASE_SPEED    = calib['base_speed']

print(f'Loaded calibration v{calib["version"]}')
print(f'  steering_gain = {STEERING_GAIN}')
print(f'  base_speed    = {BASE_SPEED}')

# Example usage inside a PD controller:
#
#   steering = angle * STEERING_GAIN + (angle - angle_last) * D_GAIN
#   left  = float(np.clip(BASE_SPEED + steering, 0.0, 1.0))
#   right = float(np.clip(BASE_SPEED - steering, 0.0, 1.0))
#   robot.left_motor.value  = left
#   robot.right_motor.value = right